# Software Engineering - Tutorial 3: The Design Patterns
## Assignment 2 - Group 2
**Instructor:** PhD. Ha Manh Hung  
**Repository Directory:** `D:\Job\thothan\VNU\Software Engineering\Assigment2`  
**Master Draw.io File:** [`assigment2_all_diagrams.drawio`](assigment2_all_diagrams.drawio) (Multi-page diagram file)

---
### 📌 Executive Summary of Assignment 2
This notebook synthesizes all solutions for **Tutorial 3 - The Design Patterns**:
1. **Exercise 1: The Singleton Design Pattern**
   - Context: Bank accounts (`CompteBancaire`) sharing a common logging system (`Journalisation`).
   - Investigation of memory redundancy (multiple logger instances).
   - Redesign with **Singleton Pattern** ensuring exactly **one** global logger instance across all accounts.
2. **Exercise 2: The Decorator Design Pattern**
   - Context: Event logging system with formatting (HTML tags) and encryption (circular character shifting).
   - Evaluation of class explosion under multiple inheritance / subclassing approaches.
   - Restructuring using the **Decorator Pattern** to allow dynamic, recursive chaining of responsibilities at runtime.

---
# Exercise 1: Bank Account & Singleton Pattern

### 1. Problem Statement
The application models a banking system where multiple bank accounts (`cb1`, `cb2`, `cb3`) record deposits and withdrawals into an activity log via `Journalisation.afficherLog()`.
In the initial unoptimized version, every `CompteBancaire` object instantiates its own `Journalisation` object in its constructor (`make_shared<Journalisation>()`), leading to memory wastage and lack of a centralized logging trace.

---
### 2. Answers to Questions

#### Question 1: Main Function Execution & Expected Output
Transactions performed across accounts to generate the required log output:
- `cb1.deposerArgent(100.0)` $\to$ Deposit of 100.000000e into account 123456789.
- `cb1.retirerArgent(80.0)` $\to$ Withdrawal of 80.000000e from account 123456789.
- `cb2.retirerArgent(50.0)` $\to$ The bank does not allow overdrafts on account 987654321 (solde = 0).
- `cb3.deposerArgent(100.0)` $\to$ Deposit of 100.000000e into account 456789012.

#### Question 2: Initial Number of Journalisation Objects
- **Answer:** **3 objects** of type `Journalisation` are created. Each time `new CompteBancaire(...)` is executed, its constructor explicitly calls `make_shared<Journalisation>()`, producing 3 distinct instances in memory.

#### Question 3: Class Diagram of Singleton Journalisation & Operation Mechanism
![Exercise 1 Singleton Class Diagram](diagram_images/ex1_singleton_diagram.png)

* **Operation Mechanism:**
  1. The default constructor `Journalisation()` is declared **private** (`-`) so that outside clients cannot instantiate objects directly with `new`.
  2. A private static member `uniqueInstance : Journalisation` holds the single created instance.
  3. A public static method `getInstance() : Journalisation` checks if `uniqueInstance == null`. If `true`, it creates the instance; otherwise, it returns the existing instance.

#### Question 4: Code Implementation
The refactored code below encapsulates the Singleton logger and modifies `CompteBancaire` to call `Journalisation.getInstance()`.

#### Question 5: Final Number of Journalisation Objects under Singleton
- **Answer:** Exactly **1 object** of type `Journalisation` is created throughout the entire execution, shared across `cb1`, `cb2`, `cb3`.

In [1]:
"""
Exercise 1: Bank Account & Singleton Logging Pattern
Tutorial 3 - Design Patterns (Software Engineering, VNU-IS)
"""
from datetime import datetime
from typing import Optional


# =====================================================================
# Singleton Pattern: Journalisation
# =====================================================================
class Journalisation:
    """
    Singleton Logger ensuring only ONE instance exists throughout the application.
    """
    _uniqueInstance: Optional["Journalisation"] = None

    def __init__(self) -> None:
        if Journalisation._uniqueInstance is not None:
            raise RuntimeError("Use Journalisation.getInstance() to access the logger instance.")
        # Simulating address/ID for verification
        self.instance_id = hex(id(self))

    @classmethod
    def getInstance(cls) -> "Journalisation":
        """Thread-safe access method to the unique Singleton instance."""
        if cls._uniqueInstance is None:
            cls._uniqueInstance = cls()
        return cls._uniqueInstance

    @classmethod
    def resetInstance(cls) -> None:
        """Utility to reset singleton instance for testing."""
        cls._uniqueInstance = None

    def afficherLog(self, log: str) -> None:
        """Prints log with current date and time formatting."""
        now = datetime.now()
        # Formatted: [19/8/2026 at 11h01] <log>
        time_str = f"[{now.day}/{now.month}/{now.year} at {now.hour:02d}h{now.minute:02d}]"
        print(f"{time_str} {log}")


# =====================================================================
# Domain Class: CompteBancaire
# =====================================================================
class CompteBancaire:
    """
    Bank account delegating operation logs to the Singleton Journalisation.
    """
    def __init__(self, numero: int) -> None:
        self.numero: int = numero
        self.solde: float = 0.0
        # Retrieves the shared unique instance from Singleton
        self.journalisation: Journalisation = Journalisation.getInstance()
        # Displaying instance address/id as in C++ original code
        print(f"BankAccount {self.numero} initialized with Logger ID: {self.journalisation.instance_id}")

    def deposerArgent(self, depot: float) -> None:
        """Deposit money into account."""
        if depot > 0.0:
            self.solde += depot
            self.journalisation.afficherLog(
                f"Deposit of {depot:.6f}e into account {self.numero}."
            )

    def retirerArgent(self, retrait: float) -> None:
        """Withdraw money from account."""
        if retrait > 0.0:
            if self.solde >= retrait:
                self.solde -= retrait
                self.journalisation.afficherLog(
                    f"Withdrawal of {retrait:.6f}e from account {self.numero}."
                )
            else:
                self.journalisation.afficherLog(
                    f"The bank does not allow overdrafts on account {self.numero}."
                )


# =====================================================================
# Main execution matching tutorial requirements (Question 1)
# =====================================================================
def main():
    print("--- 1. Creating Bank Accounts cb1, cb2, cb3 ---")
    cb1 = CompteBancaire(123456789)
    cb2 = CompteBancaire(987654321)
    cb3 = CompteBancaire(456789012)

    print("\n--- 2. Executing Transactions to match Question 1 Output ---")
    # Using cb1: deposit 100, then withdraw 80
    cb1.deposerArgent(100.0)
    cb1.retirerArgent(80.0)

    # Using cb2: attempt overdraft (solde is 0, withdrawal 50 -> rejected)
    cb2.retirerArgent(50.0)

    # Using cb3: deposit 100
    cb3.deposerArgent(100.0)


if __name__ == "__main__":
    main()

--- 1. Creating Bank Accounts cb1, cb2, cb3 ---
BankAccount 123456789 initialized with Logger ID: 0x120c2f46f90
BankAccount 987654321 initialized with Logger ID: 0x120c2f46f90
BankAccount 456789012 initialized with Logger ID: 0x120c2f46f90

--- 2. Executing Transactions to match Question 1 Output ---
[14/10/2026 at 17h24] Deposit of 100.000000e into account 123456789.
[14/10/2026 at 17h24] Withdrawal of 80.000000e from account 123456789.
[14/10/2026 at 17h24] The bank does not allow overdrafts on account 987654321.
[14/10/2026 at 17h24] Deposit of 100.000000e into account 456789012.


---
# Exercise 2: Event Logging & Decorator Pattern

### 1. Problem Statement
We want to keep track of abnormal application events by logging messages to a console or saving them to a file. In addition, messages may need to be:
- **Formatted in HTML:** `<HTML><BODY> <b> <msg> </b> </BODY> </HTML>`
- **Encrypted:** Circularly shifting each character $n$ positions to the left (e.g., `'Message d'erreur n°1'` shifted 4 positions becomes `'age d'erreur n°1Mess'`).

---
### 2. Answers to Questions

#### Question 1: Comment on the Proposed Solution (Figure 2)
The initial solution in Figure 2 uses classical inheritance to create specialized subclasses for every combination:
- `HTMLConsoleLogger`, `EncryptConsoleLogger`
- `HTMLFileLogger`, `EncryptFileLogger`

* **Criticisms / Drawbacks:**
  1. **Class Explosion (Combinatorial Explosion):** If we introduce $M$ log destinations (Console, File, Database, Network) and $N$ transformations (HTML, Encrypt, Timestamp, Compress), we must create $M \times 2^N$ separate classes!
  2. **Rigid Static Binding:** Responsibilities are fixed at compile-time and cannot be added or removed dynamically at runtime.
  3. **No Support for Multiple Chaining:** Figure 2 cannot handle combined operations (e.g., both HTML formatting **AND** encryption) without inventing even more hybrid subclasses like `HTMLEncryptFileLogger`.

#### Question 2: Proposed Class Diagram using Decorator Pattern
![Exercise 2 Decorator Class Diagram](diagram_images/ex2_decorator_class_diagram.png)

- **`Logger` (Interface Component):** Declares `log(msg)`.
- **`ConsoleLogger`, `FileLogger` (Concrete Components):** Terminal output destinations.
- **`DecorateurLogger` (Base Decorator):** Implements `Logger` and holds an internal `1 -logger_: Logger` reference.
- **`HTMLLogger`, `EncryptLogger` (Concrete Decorators):** Apply formatting/encryption before delegating to the wrapped `logger_`.

#### Question 3: Object Diagram (`HTMLLogger` wrapping `FileLogger`)
See the left panel below: `hLogger: HTMLLogger` holds a pointer/reference to `fLogger: FileLogger`.

#### Question 4: Sequence Diagram (Storing HTML Log in File)
See the right panel below:
1. `Client` calls `hLogger.log("Message d'erreur n°1")`.
2. `hLogger` executes internal `formaterHTML(msg)`.
3. `hLogger` calls `fLogger.log("<HTML><BODY> <b> ... </b> </BODY> </HTML>")`.
4. `fLogger` appends formatted string to `nomFichier_`.

![Exercise 2 Object and Sequence Diagram](diagram_images/ex2_object_sequence_diagrams.png)

#### Questions 5 & 6: Python Implementation & Multi-Case Testing
The complete executable solution below validates:
- Case 1: `EncryptLogger` attached to `ConsoleLogger` (10 messages).
- Case 2: `HTMLLogger` attached to `FileLogger` (10 messages into `FichierLog1.html`).
- Case 3: Chained decoration (`HTMLLogger` $\to$ `EncryptLogger` $\to$ `FileLogger`).

In [2]:
"""
Exercise 2: Logging System & Decorator Design Pattern
Tutorial 3 - Design Patterns (Software Engineering, VNU-IS)
"""
from abc import ABC, abstractmethod
import os


# =====================================================================
# 1. Component Interface
# =====================================================================
class Logger(ABC):
    """Abstract Component defining common interface for all loggers."""

    @abstractmethod
    def log(self, msg: str) -> None:
        pass


# =====================================================================
# 2. Concrete Components
# =====================================================================
class ConsoleLogger(Logger):
    """Logs messages directly to the standard console output."""

    def log(self, msg: str) -> None:
        print(msg)


class FileLogger(Logger):
    """Logs messages by appending them to a specified file."""

    def __init__(self, nomFichier: str) -> None:
        self.nomFichier_ = nomFichier

    def log(self, msg: str) -> None:
        try:
            with open(self.nomFichier_, "a", encoding="utf-8") as f:
                f.write(msg + "\n")
        except Exception as e:
            print(f"Erreur : impossible d'ouvrir le fichier {self.nomFichier_} : {e}")


# =====================================================================
# 3. Base Decorator
# =====================================================================
class DecorateurLogger(Logger):
    """
    Abstract Decorator maintaining a reference to a Logger component
    and conforming to the Logger interface.
    """

    def __init__(self, logger: Logger) -> None:
        self.logger_ = logger

    def log(self, msg: str) -> None:
        # Default forwarding behavior
        self.logger_.log(msg)


# =====================================================================
# 4. Concrete Decorators
# =====================================================================
class HTMLLogger(DecorateurLogger):
    """
    Decorates log messages by wrapping them inside HTML tags:
    <HTML><BODY> <b> <msg> </b> </BODY> </HTML>
    """

    def formaterHTML(self, msg: str) -> str:
        return f"<HTML><BODY> <b> {msg} </b> </BODY> </HTML>"

    def log(self, msg: str) -> None:
        html_msg = self.formaterHTML(msg)
        self.logger_.log(html_msg)


class EncryptLogger(DecorateurLogger):
    """
    Decorates log messages by circularly shifting each character
    n positions to the left (default n = 4).
    E.g. 'Message d'erreur n°1' shifted 4 positions -> 'age d'erreur n°1Mess'
    """

    def __init__(self, logger: Logger, shift: int = 4) -> None:
        super().__init__(logger)
        self.shift = shift

    def crypter(self, msg: str) -> str:
        if not msg:
            return msg
        n = self.shift % len(msg)
        return msg[n:] + msg[:n]

    def log(self, msg: str) -> None:
        encrypted_msg = self.crypter(msg)
        self.logger_.log(encrypted_msg)


# =====================================================================
# Main execution matching tutorial requirements (Question 6)
# =====================================================================
def main():
    print("=== Case 1: EncryptLogger attached to ConsoleLogger ===")
    cLogger1 = ConsoleLogger()
    eLogger1 = EncryptLogger(cLogger1, shift=4)

    # Testing specific tutorial example first:
    test_str = "Message d'erreur n°1"
    print(f"Sample test: '{test_str}' -> '{eLogger1.crypter(test_str)}'")

    print("\nGenerating 10 encrypted log messages to console:")
    for i in range(10):
        eLogger1.log(f"{i} : A message to log")

    print("\n=== Case 2: HTMLLogger attached to FileLogger ===")
    output_filename = "FichierLog1.html"
    # Ensure fresh file
    if os.path.exists(output_filename):
        os.remove(output_filename)

    fLogger1 = FileLogger(output_filename)
    hLogger1 = HTMLLogger(fLogger1)

    print(f"Generating 10 HTML formatted messages into '{output_filename}'...")
    for i in range(10):
        hLogger1.log(f"{i} : A message to log")

    print("File content preview:")
    with open(output_filename, "r", encoding="utf-8") as f:
        print(f.read().strip())

    print("\n=== Case 3: Composed Chaining (HTML + Encryption + FileLogger) ===")
    combo_file = "FichierLog_Combo.html"
    if os.path.exists(combo_file):
        os.remove(combo_file)
    comboLogger = HTMLLogger(EncryptLogger(FileLogger(combo_file)))
    comboLogger.log("Critical Error Occurred")
    with open(combo_file, "r", encoding="utf-8") as f:
        print("Combo output:", f.read().strip())


if __name__ == "__main__":
    main()

=== Case 1: EncryptLogger attached to ConsoleLogger ===
Sample test: 'Message d'erreur n°1' -> 'age d'erreur n°1Mess'

Generating 10 encrypted log messages to console:
A message to log0 : 
A message to log1 : 
A message to log2 : 
A message to log3 : 
A message to log4 : 
A message to log5 : 
A message to log6 : 
A message to log7 : 
A message to log8 : 
A message to log9 : 

=== Case 2: HTMLLogger attached to FileLogger ===
Generating 10 HTML formatted messages into 'FichierLog1.html'...
File content preview:
<HTML><BODY> <b> 0 : A message to log </b> </BODY> </HTML>
<HTML><BODY> <b> 1 : A message to log </b> </BODY> </HTML>
<HTML><BODY> <b> 2 : A message to log </b> </BODY> </HTML>
<HTML><BODY> <b> 3 : A message to log </b> </BODY> </HTML>
<HTML><BODY> <b> 4 : A message to log </b> </BODY> </HTML>
<HTML><BODY> <b> 5 : A message to log </b> </BODY> </HTML>
<HTML><BODY> <b> 6 : A message to log </b> </BODY> </HTML>
<HTML><BODY> <b> 7 : A message to log </b> </BODY> </HTML>
<HTML><BODY>